In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

In [ ]:
df = pd.read_csv("/content/kfc_sales_raw.csv")

In [ ]:
df.head()

,order_id,order_date,store_id,city,country,product,category,quantity,unit_price,total_amount,payment_method,order_type,customer_age,rating
0,ORD-142586,2025/12/18 18:04,KFC-007,London,UK,Pepsi (Large),Beverages,1.0,220.0,220.0,Card,Takeaway,20.0,3.6
1,ORD-130648,2024-02-27 11:43:00,KFC-005,Peshawar,Pakistan,Chicky Meal,Meals,2.0,750.0,1500.0,Cash,dine-in,61.0,2.4
2,ORD-107247,2024-09-15 18:21:00,KFC-007,London,UK,Chicken Bucket (8 pcs),Buckets,3.0,1850.0,5550.0,cash,Dine-In,61.0,4.1
3,ORD-178030,09-18-2023,KFC-006,Dubai,UAE,ZINGER BURGER,Burgers,2.0,650.0,1300.0,cash,Takeaway,27.0,4.0
4,ORD-127369,2024-07-13 13:04:00,KFC-003,Karachi,Pakistan,Mighty Zinger,Burgers,5.0,890.0,4450.0,NaN,dine-in,58.0,4.7


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48205 entries, 0 to 48204
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        48205 non-null  object 
 1   order_date      48031 non-null  object 
 2   store_id        48204 non-null  object 
 3   city            46313 non-null  object 
 4   country         48204 non-null  object 
 5   product         48204 non-null  object 
 6   category        47218 non-null  object 
 7   quantity        48204 non-null  float64
 8   unit_price      48204 non-null  float64
 9   total_amount    48204 non-null  float64
 10  payment_method  46754 non-null  object 
 11  order_type      46749 non-null  object 
 12  customer_age    45768 non-null  float64
 13  rating          44375 non-null  float64
dtypes: float64(5), object(9)
memory usage: 5.1+ MB


In [ ]:
df.describe()

,quantity,unit_price,total_amount,customer_age,rating
count,48204.000000,48204.000000,4.820400e+04,45768.000000,44375.000000
mean,9.015621,653.900714,6.251546e+03,40.041841,2.999006
std,91.565523,496.457643,8.109017e+04,25.438083,1.168395
min,1.000000,150.000000,-9.250000e+03,-5.000000,-1.000000
25%,2.000000,220.000000,6.600000e+02,27.000000,2.000000
50%,3.000000,520.000000,1.300000e+03,40.000000,3.000000
75%,4.000000,890.000000,2.600000e+03,52.000000,4.000000
max,1998.000000,1850.000000,3.557550e+06,999.000000,10.000000


In [ ]:
df.isnull().sum()

,0
order_id,0
order_date,174
store_id,1
city,1892
country,1
product,1
category,987
quantity,1
unit_price,1
total_amount,1


In [ ]:
df.duplicated().sum()

np.int64(427)

In [ ]:

df_clean = df.copy()

In [ ]:
df_clean.shape

(48205, 14)

In [ ]:
raw= "kfc_sales_raw.csv"
clean = "kfc_sales_clean.csv"



In [ ]:
original_rows = len(df)

# Standardizing text column

In [ ]:
str_cols = df.select_dtypes(include="object").columns
for c in str_cols:
    df[c] = df[c].str.strip()

# 2b. Standardize store_id
df["store_id"] = df["store_id"].str.upper()

In [ ]:
city_map = {
    "islamabad": "Islamabad", "ISLAMABAD": "Islamabad", "Islambad": "Islamabad",
    "lahore": "Lahore", "LAHORE": "Lahore", "Lahor": "Lahore",
    "karachi": "Karachi", "KARACHI": "Karachi", "Karrachi": "Karachi",
    "rawalpindi": "Rawalpindi", "Pindi": "Rawalpindi",
}
df["city"] = df["city"].replace(city_map)

In [ ]:
product_map = {
    "zinger burger": "Zinger Burger", "Zinger burger": "Zinger Burger",
    "ZINGER BURGER": "Zinger Burger", "Zinger Burgr": "Zinger Burger",
    "Hot Wings": "Hot Wings (5 pcs)", "hot wings (5 pcs)": "Hot Wings (5 pcs)",
    "HotWings": "Hot Wings (5 pcs)",
    "Fries Regular": "Fries (Regular)", "fries (regular)": "Fries (Regular)",
    "Regular Fries": "Fries (Regular)",
}
df["product"] = df["product"].replace(product_map)

In [ ]:
payment_map = {
    "cash": "Cash", "CASH": "Cash",
    "card": "Card", "CARD": "Card", "Credit Card": "Card",
    "online": "Online", "ONLINE": "Online",
}
df["payment_method"] = df["payment_method"].replace(payment_map)


In [ ]:
order_map = {
    "dine-in": "Dine-In", "DINE-IN": "Dine-In",
    "takeaway": "Takeaway", "TAKEAWAY": "Takeaway",
    "delivery": "Delivery", "DELIVERY": "Delivery",
}
df["order_type"] = df["order_type"].replace(order_map)

print("Unique cities now:", sorted(df["city"].dropna().unique()))
print("\n" + "=" * 60)

Unique cities now: ['Dubai', 'Islamabad', 'Karachi', 'Lahore', 'London', 'Manchester', 'Peshawar', 'Rawalpindi']



# Filling Missing values

fillling missing values by iditifying relationship between columns ..  like by store id we can identity city

In [ ]:
store_city = (
    df.dropna(subset=["city"])
      .groupby("store_id")["city"]
      .agg(lambda s: s.value_counts().index[0])
      .to_dict()
)

In [ ]:
missing_city_before = df["city"].isna().sum()
df["city"] = df.apply(
    lambda r: store_city.get(r["store_id"], r["city"]) if pd.isna(r["city"]) else r["city"],
    axis=1,
)
print(f"City: filled {missing_city_before - df['city'].isna().sum()} values from store_id")


City: filled 1891 values from store_id


In [ ]:
prod_cat = (
    df.dropna(subset=["category"])
      .groupby("product")["category"]
      .agg(lambda s: s.value_counts().index[0])
      .to_dict()
)

In [ ]:
missing_cat_before = df["category"].isna().sum()
df["category"] = df.apply(
    lambda r: prod_cat.get(r["product"], r["category"]) if pd.isna(r["category"]) else r["category"],
    axis=1,
)
print(f"Category: filled {missing_cat_before - df['category'].isna().sum()} values from product")
print("\n" + "=" * 60)

Category: filled 986 values from product



# Handling outliers

we will fill impossible values with NAN   because data contain outliers like 999 years age

In [ ]:
invalid_age = ~df["customer_age"].between(5, 100)
n_bad_age = (invalid_age & df["customer_age"].notna()).sum()
df.loc[invalid_age, "customer_age"] = np.nan
print(f"Age: set {n_bad_age} impossible ages to NaN")

Age: set 92 impossible ages to NaN


In [ ]:
invalid_rating = ~df["rating"].between(1, 5)
n_bad_rating = (invalid_rating & df["rating"].notna()).sum()
df.loc[invalid_rating, "rating"] = np.nan
print(f"Rating: set {n_bad_rating} out-of-range ratings to NaN")


Rating: set 97 out-of-range ratings to NaN


In [ ]:
invalid_qty = ~df["quantity"].between(1, 50)
n_bad_qty = invalid_qty.sum()
df = df[~invalid_qty].copy()
print(f"Quantity: dropped {n_bad_qty} rows with impossible quantities")


Quantity: dropped 232 rows with impossible quantities


In [ ]:
neg_total = df["total_amount"] < 0
n_neg = neg_total.sum()
df = df[~neg_total].copy()
print(f"Total: dropped {n_neg} rows with negative total_amount")
print("\n" + "=" * 60)


Total: dropped 133 rows with negative total_amount



filling unrecoverable entities

In [ ]:
df["payment_method"] = df["payment_method"].fillna("Unknown")
df["order_type"] = df["order_type"].fillna("Unknown")
print("payment_method / order_type: filled nulls with 'Unknown'")

payment_method / order_type: filled nulls with 'Unknown'


In [ ]:
age_median = df["customer_age"].median()
n_age_null = df["customer_age"].isna().sum()
df["customer_age"] = df["customer_age"].fillna(age_median).round().astype(int)
print(f"customer_age: filled {n_age_null} nulls with median ({age_median:.0f})")


customer_age: filled 2516 nulls with median (40)


In [ ]:
print(f"rating: kept {df['rating'].isna().sum()} nulls (not imputed)")
print("\n" + "=" * 60)

rating: kept 3896 nulls (not imputed)



# Handling Duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} full duplicate rows")


Removed 518 full duplicate rows


In [ ]:
before = len(df)
df = df.drop_duplicates(subset="order_id", keep="first")
print(f"Removed {before - len(df)} rows with duplicate order_id")
print("\n" + "=" * 60)


Removed 417 rows with duplicate order_id



Handling missing total amount valuues

In [ ]:
expected = df["quantity"] * df["unit_price"]
mismatch = (df["total_amount"] != expected).sum()
df["total_amount"] = expected  # trust qty*price as source of truth
print(f"Corrected {mismatch} rows where total didn't match quantity * unit_price")
print("\n" + "=" * 60)


Corrected 520 rows where total didn't match quantity * unit_price



In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"], format="mixed", errors="coerce")
n_bad_dates = df["order_date"].isna().sum()
df = df.dropna(subset=["order_date"])
print(f"Dropped {n_bad_dates} rows with missing/unparseable dates")


Dropped 173 rows with missing/unparseable dates


In [ ]:
df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.month
df["day_name"] = df["order_date"].dt.day_name()
df["hour"] = df["order_date"].dt.hour
print("Added year, month, day_name, hour columns")
print("\n" + "=" * 60)

Added year, month, day_name, hour columns



In [ ]:

print(f"Rows: {original_rows:,} (raw) -> {len(df):,} (clean)")
print(f"Removed {original_rows - len(df):,} rows "
      f"({(original_rows - len(df)) / original_rows * 100:.1f}%)")
print("\nRemaining missing values (only 'rating' should remain):")
print(df.isna().sum())

Rows: 48,205 (raw) -> 46,732 (clean)
Removed 1,473 rows (3.1%)

Remaining missing values (only 'rating' should remain):
order_id             0
order_date           0
store_id             0
city                 0
country              0
product              0
category             0
quantity             0
unit_price           0
total_amount         0
payment_method       0
order_type           0
customer_age         0
rating            3817
year                 0
month                0
day_name             0
hour                 0
dtype: int64


In [ ]:
df.to_csv(CLEAN, index=False)
print(f"\nSaved cleaned data -> {CLEAN}")


Saved cleaned data -> kfc_sales_clean.csv


In [ ]:
CLEAN = "kfc_sales_clean.csv"
df = pd.read_csv(CLEAN, parse_dates=["order_date"])


In [ ]:
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
KFC_RED = "#A6192E"

def money(x, pos=None):
    """Format numbers as PKR with thousands separators."""
    return f"{x:,.0f}"

print("=" * 60)
print("KFC SALES — EXPLORATORY DATA ANALYSIS")
print("=" * 60)
print(f"Rows: {len(df):,}")
print(f"Date range: {df['order_date'].min().date()} to {df['order_date'].max().date()}")


KFC SALES — EXPLORATORY DATA ANALYSIS
Rows: 46,732
Date range: 2023-01-01 to 2025-12-30


In [ ]:
print("\n" + "=" * 60)
print("2. HEADLINE KPIs")
print("=" * 60)

total_revenue = df["total_amount"].sum()
total_orders = df["order_id"].nunique()
total_items = df["quantity"].sum()
avg_order_value = df.groupby("order_id")["total_amount"].sum().mean()
avg_rating = df["rating"].mean()          # nulls automatically ignored
rating_response = df["rating"].notna().mean() * 100

print(f"Total revenue:        PKR {total_revenue:,.0f}")
print(f"Total orders:         {total_orders:,}")
print(f"Total items sold:     {total_items:,}")
print(f"Avg order value:      PKR {avg_order_value:,.0f}")
print(f"Avg rating:           {avg_rating:.2f} / 5  (from {rating_response:.0f}% of orders that rated)")


2. HEADLINE KPIs
Total revenue:        PKR 91,786,860
Total orders:         46,732
Total items sold:     140,433.0
Avg order value:      PKR 1,964
Avg rating:           3.00 / 5  (from 92% of orders that rated)


In [ ]:
print("\n" + "=" * 60)
print("3. SALES OVER TIME")
print("=" * 60)

monthly = (
    df.set_index("order_date")
      .resample("ME")["total_amount"]
      .sum()
)
print("Monthly revenue (first 6 months):")
print(monthly.head(6).apply(lambda v: f"PKR {v:,.0f}"))

fig, ax = plt.subplots()
ax.plot(monthly.index, monthly.values, color=KFC_RED, linewidth=2)
ax.set_title("Monthly Revenue Trend")
ax.set_ylabel("Revenue (PKR)")

plt.tight_layout()
plt.savefig("eda_01_monthly_trend.png", dpi=120)
plt.close()


3. SALES OVER TIME
Monthly revenue (first 6 months):
order_date
2023-01-31    PKR 2,557,520
2023-02-28    PKR 2,367,000
2023-03-31    PKR 2,620,710
2023-04-30    PKR 2,410,500
2023-05-31    PKR 2,517,310
2023-06-30    PKR 2,667,690
Freq: ME, Name: total_amount, dtype: object


In [ ]:
yearly = df.groupby("year")["total_amount"].sum()
print("\nRevenue by year:")
print(yearly.apply(lambda v: f"PKR {v:,.0f}"))



Revenue by year:
year
2023    PKR 30,420,960
2024    PKR 30,904,010
2025    PKR 30,461,890
Name: total_amount, dtype: object


In [ ]:
print("\n" + "=" * 60)
print("4. PRODUCT & CATEGORY PERFORMANCE")
print("=" * 60)
prod_rev = df.groupby("product")["total_amount"].sum().sort_values(ascending=False)
print("Top 5 products by revenue:")
print(prod_rev.head(5).apply(lambda v: f"PKR {v:,.0f}"))
print("\nBottom 5 products by revenue:")
print(prod_rev.tail(5).apply(lambda v: f"PKR {v:,.0f}"))



4. PRODUCT & CATEGORY PERFORMANCE
Top 5 products by revenue:
product
Chicken Bucket (8 pcs)    PKR 17,208,700
Boneless Bucket           PKR 15,460,500
Dinner Meal                PKR 9,375,300
Mighty Zinger              PKR 8,011,780
Chicky Meal                PKR 7,084,500
Name: total_amount, dtype: object

Bottom 5 products by revenue:
product
Fries (Large)      PKR 2,878,720
Fries (Regular)    PKR 2,092,640
Pepsi (Large)      PKR 2,011,680
Coleslaw           PKR 1,670,940
Pepsi (Regular)    PKR 1,417,500
Name: total_amount, dtype: object


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
prod_rev.head(10).sort_values().plot(kind="barh", color=KFC_RED, ax=ax)
ax.set_title("Top 10 Products by Revenue")
ax.set_xlabel("Revenue (PKR)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(money))
plt.tight_layout()
plt.savefig("eda_02_top_products.png", dpi=120)
plt.close()


In [ ]:
rated = df[df["rating"].notna()]
print(f"\nRatings: {len(rated):,} orders rated ({len(rated)/len(df)*100:.0f}%)")
print(f"Avg rating by category:")
print(rated.groupby("category")["rating"].mean().sort_values(ascending=False).round(2))

fig, ax = plt.subplots()
ax.hist(rated["rating"], bins=np.arange(1, 5.5, 0.5), color=KFC_RED, edgecolor="white")
ax.set_title("Rating Distribution (rated orders only)")
ax.set_xlabel("Rating")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig("eda_06_rating_dist.png", dpi=120)
plt.close()


Ratings: 42,915 orders rated (92%)
Avg rating by category:
category
Chicken      3.04
Burgers      3.01
Meals        3.01
Beverages    3.00
Sides        2.99
Rice         2.99
Wraps        2.97
Buckets      2.96
Name: rating, dtype: float64


In [ ]:
hour_orders = df.groupby("hour")["order_id"].count()
print("Busiest hours (top 5):")
print(hour_orders.sort_values(ascending=False).head(5))


Busiest hours (top 5):
hour
0     3581
20    3396
16    3370
12    3364
13    3340
Name: order_id, dtype: int64


In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
day_rev = df.groupby("day_name")["total_amount"].sum().reindex(day_order)
print("\nRevenue by day of week:")
print(day_rev.apply(lambda v: f"PKR {v:,.0f}"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
hour_orders.plot(kind="line", marker="o", color=KFC_RED, ax=axes[0])
axes[0].set_title("Orders by Hour of Day")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("Orders")

day_rev.plot(kind="bar", color=KFC_RED, ax=axes[1])
axes[1].set_title("Revenue by Day of Week")
axes[1].set_ylabel("Revenue (PKR)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(money))
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("eda_07_time_patterns.png", dpi=120)
plt.close()


Revenue by day of week:
day_name
Monday       PKR 12,764,480
Tuesday      PKR 12,722,360
Wednesday    PKR 13,039,250
Thursday     PKR 13,191,390
Friday       PKR 13,017,810
Saturday     PKR 13,653,120
Sunday       PKR 13,398,450
Name: total_amount, dtype: object


In [ ]:
prod_rev = df.groupby("product")["total_amount"].sum().sort_values(ascending=False)
cat_rev = df.groupby("category")["total_amount"].sum().sort_values(ascending=False)
store_rev = df.groupby(["store_id", "city"])["total_amount"].sum().sort_values(ascending=False)
hour_orders = df.groupby("hour")["order_id"].count()
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
day_rev = df.groupby("day_name")["total_amount"].sum().reindex(day_order)
total_revenue = df["total_amount"].sum()
avg_order_value = df.groupby("order_id")["total_amount"].sum().mean()

print("=" * 60)
print("9. KEY TAKEAWAYS")
print("=" * 60)
print(f"- Best-selling product:   {prod_rev.index[0]} (PKR {prod_rev.iloc[0]:,.0f})")
print(f"- Worst-selling product:  {prod_rev.index[-1]} (PKR {prod_rev.iloc[-1]:,.0f})")
print(f"- Top store:              {store_rev.index[0][0]} in {store_rev.index[0][1]}")
print(f"- Busiest hour:           {hour_orders.idxmax()}:00")
print(f"- Best day:               {day_rev.idxmax()}")
print(f"- Avg order value:        PKR {avg_order_value:,.0f}")

9. KEY TAKEAWAYS
- Best-selling product:   Chicken Bucket (8 pcs) (PKR 17,208,700)
- Worst-selling product:  Pepsi (Regular) (PKR 1,417,500)
- Top store:              KFC-002 in Lahore
- Busiest hour:           0:00
- Best day:               Saturday
- Avg order value:        PKR 1,964


In [ ]:
# Save the cleaned DataFrame to a CSV in Colab's temporary storage
df.to_csv("kfc_sales_clean.csv", index=False)

# Trigger a download to your computer
from google.colab import files
files.download("kfc_sales_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>